## Init

In [ ]:
# ---------------- Imports ----------------
import os

from datetime import datetime

import torch
import yaml

from sentence_transformers import SentenceTransformer

from elicitation.metrics import progression, turn_length_ratio
from elicitation.metrics.utils import load_dialogues


In [ ]:
# ---------------- Args ----------------
# Paths
model_choice_name = "sentence-transformers/all-MiniLM-L12-v2"
dataset_choice_name = "yield-v1-small1pct"


# Constants
PROGRESSION_K = 2 
PROGRESSION_GAMMA = 0.9




In [ ]:
# ---------------- Config ----------------
timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj_store"]

data_path = os.path.join("./sample_data")

dataset_path = os.path.join(data_path, dataset_choice_name)

models_folderpath = config["paths"]["models"]

model_choice = os.path.join(models_folderpath, model_choice_name)


save_folder_path = os.path.join(proj_store, "evaluation", "interaction-metrics", f"{dataset_choice_name}")
os.makedirs(save_folder_path, exist_ok=True)

# Load sentence embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
embedding_model = SentenceTransformer(model_choice, device=device)




cuda


In [8]:
all_dialogues = list(load_dialogues(dataset_path))

print("Loaded", len(all_dialogues), "dialogues")



Loaded 26 dialogues


## Progression

In [9]:
progression_df = progression(
    dialogues=all_dialogues, 
    embedding_model=embedding_model, 
    device=device, 
    k=PROGRESSION_K, 
    gamma=PROGRESSION_GAMMA
)

display(progression_df)



Progression: 100%|██████████| 26/26 [00:02<00:00, 12.78it/s]


,all,dialogues,progression
0,all,26,0.700819


In [10]:
progression_df = progression(
    dialogues=all_dialogues, 
    embedding_model=embedding_model, 
    device=device, 
    k=PROGRESSION_K, 
    gamma=PROGRESSION_GAMMA, 
    group_by="domain", 
    sort_by="domain",
    export_raw=True,
)

display(progression_df.head())


Progression: 100%|██████████| 26/26 [00:01<00:00, 14.20it/s]


,domain,dialogue_id,pair_index,progression_score
0,academic_interviews,covid-19-threshold-00011,72,0.439533
1,academic_interviews,covid-19-threshold-00011,116,0.763844
2,academic_interviews,covid-19-threshold-00011,117,0.827594
3,academic_interviews,covid-19-threshold-00011,118,0.733332
4,academic_interviews,covid-19-threshold-00011,119,0.576964


In [11]:
progression_df = progression(
    dialogues=all_dialogues, 
    embedding_model=embedding_model, 
    device=device, 
    k=PROGRESSION_K, 
    gamma=PROGRESSION_GAMMA, 
    group_by="domain", 
    sort_by="domain"
)

display(progression_df)



Progression: 100%|██████████| 26/26 [00:01<00:00, 14.13it/s]


,domain,dialogues,progression
0,academic_interviews,3,0.726250
1,journalistic_investigations,3,0.617804
2,judicial_proceedings,7,0.729637
3,oral_history,13,0.698591


In [12]:
# Save to CSV
progression_df.to_csv(os.path.join(save_folder_path, f"progression.csv"), index=False)



## Turn-Length Ratio

In [17]:
turn_length_df = turn_length_ratio(
    dialogues=all_dialogues, 
)

display(turn_length_df)



Turn Length: 100%|██████████| 26/26 [00:00<00:00, 2728.82it/s]


,All,elicitor_avg_tokens,respondent_avg_tokens,turn_length_ratio
0,All,27.31,111.04,4.065


In [18]:
turn_length_df = turn_length_ratio(
    dialogues=all_dialogues, 
    group_by="domain", 
    sort_by="domain",
    export_raw=True,
)

display(turn_length_df.head())

Turn Length: 100%|██████████| 26/26 [00:00<00:00, 2844.78it/s]


,domain,dialogue_id,elicitor_avg_tokens,respondent_avg_tokens,turn_length_ratio,elicitor_turns,respondent_turns
0,academic_interviews,ai-feedback-moving-beyond-00004,22.17,33.10,1.493,64,63
1,academic_interviews,covid-19-threshold-00011,10.71,56.76,5.300,62,62
2,academic_interviews,drivers-of-food-choice-00003,10.95,11.19,1.021,107,107
3,journalistic_investigations,voa-news-00025,29.39,82.00,2.790,18,18
4,journalistic_investigations,voa-news-00037,33.78,332.44,9.842,9,9


In [19]:
turn_length_df = turn_length_ratio(
    dialogues=all_dialogues, 
    group_by="domain", 
    sort_by="domain"
)

display(turn_length_df)



Turn Length: 100%|██████████| 26/26 [00:00<00:00, 2881.08it/s]


,domain,elicitor_avg_tokens,respondent_avg_tokens,turn_length_ratio
0,academic_interviews,13.97,29.31,2.098
1,journalistic_investigations,37.55,147.38,3.925
2,judicial_proceedings,31.94,55.46,1.736
3,oral_history,21.55,239.69,11.121


In [20]:
# Save to CSV
turn_length_df.to_csv(os.path.join(save_folder_path, f"turn_length_ratio.csv"), index=False)

